In [0]:
"""
02_silver_transformation.py

Purpose:
--------
This notebook implements the Silver layer
of the Medallion Architecture.

The notebook performs:
1. Standardization of column names
2. Data type casting
3. JSON payload parsing
4. Payload flattening
5. Data cleansing
6. Creation of analytics-ready Silver tables

Execution Order:
----------------
1. 01_bronze_ingestion
2. 02_silver_transformation
3. 03_gold_datamart

Assumptions:
------------
- event_payload contains JSON data
- parameter_value may contain item_id when parameter_name = 'item_id'
- event_name represents event_type
- event_time is a valid timestamp
"""

# --------------------------------------------------
# Import Required Libraries
# --------------------------------------------------

import re
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    to_timestamp,
    year,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

# --------------------------------------------------
# Read Bronze Tables
# --------------------------------------------------
# Bronze tables are used as input
# for Silver transformations.
# --------------------------------------------------

item_df = spark.table("bronze_item")

event_df = spark.table("bronze_event")

# --------------------------------------------------
# DDL Statements
# --------------------------------------------------
# Explicit DDLs are included to satisfy
# assignment requirements.
# --------------------------------------------------

spark.sql("DROP TABLE IF EXISTS silver_item")

spark.sql("""

CREATE TABLE IF NOT EXISTS silver_item (

    id STRING,
    adjective STRING,
    category STRING,
    created_at STRING,
    modifier STRING,
    name STRING,
    price STRING

)

USING DELTA

""")

spark.sql("DROP TABLE IF EXISTS silver_event")

spark.sql("""

CREATE TABLE IF NOT EXISTS silver_event (
    event_id STRING,
    event_time TIMESTAMP,
    user_id STRING,
    year INT,
    event_type STRING,
    platform STRING,
    parameter_name STRING,
    parameter_value STRING,
    item_id STRING
)

USING DELTA
PARTITIONED BY (year)

""")

# --------------------------------------------------
# Helper Functions
# --------------------------------------------------
# Helper functions are used to standardize
# column naming conventions.
# --------------------------------------------------

def to_snake_case(name):

    name = re.sub(
        r'([a-z0-9])([A-Z])',
        r'\\1_\\2',
        name
    )

    return name.lower()

def standardize_columns(df):

    for c in df.columns:

        df = df.withColumnRenamed(
            c,
            to_snake_case(c)
        )

    return df

# --------------------------------------------------
# Standardize Column Names
# --------------------------------------------------
# Snake case naming convention is used
# for consistency and maintainability.
# --------------------------------------------------

item_df = standardize_columns(item_df)

# --------------------------------------------------
# Fix "id" column value from decimal like values
# --------------------------------------------------

item_df = item_df.withColumn(
    "id",
    F.regexp_replace(
        F.trim(F.col("id")),
        "\\.0$",
        ""
    )
)
# --------------------------------------------------
# Explore Source Schemas
# --------------------------------------------------
# Schema inspection is performed before
# transformations and type casting.
# --------------------------------------------------

print("Item Schema:")

item_df.printSchema()

print("Event Schema:")

event_df.printSchema()

# --------------------------------------------------
# Cast Event Timestamp
# --------------------------------------------------
# event_time is converted from STRING
# to TIMESTAMP type.
# --------------------------------------------------

event_df = (

    event_df
    .withColumn(
        "event_time",
        to_timestamp(col("event_time"))
    )

)

# --------------------------------------------------
# Add Year Column
# --------------------------------------------------
# Year is extracted for:
# - partitioning
# - yearly analytics
# - Gold layer aggregations
# --------------------------------------------------

event_df = event_df.withColumn(

    "year",

    year(col("event_time"))

)

# --------------------------------------------------
# Payload Parsing and Flattening
# --------------------------------------------------
# event_payload contains semi-structured
# JSON data.
#
# Example:
#
# {
#   "event_name":"view_item",
#   "platform":"android",
#   "parameter_name":"item_id",
#   "parameter_value":"3526"
# }
#
# The payload is parsed and flattened into
# analytics-ready columns.
# --------------------------------------------------

# --------------------------------------------------
# Define Payload Schema
# --------------------------------------------------
# Explicit schema definition improves:
# - parsing reliability
# - performance
# - schema consistency
# --------------------------------------------------

payload_schema = StructType([

    StructField(
        "event_name",
        StringType(),
        True
    ),
    StructField(
        "platform",
        StringType(),
        True
    ),
    StructField(
        "parameter_name",
        StringType(),
        True
    ),
    StructField(
        "parameter_value",
        StringType(),
        True
    )
])

# --------------------------------------------------
# Parse JSON Payload
# --------------------------------------------------
# event_payload is converted from
# JSON STRING into STRUCT type.
# --------------------------------------------------

event_df = event_df.withColumn(

    "payload_json",

    from_json(
        col("event_payload"),
        payload_schema
    )

)

# --------------------------------------------------
# Flatten Payload Fields
# --------------------------------------------------
# Nested payload fields are flattened
# into top-level analytical columns.
# --------------------------------------------------

event_df = event_df.select(

    "*",
    col("payload_json.event_name").alias(
        "event_type"
    ),
    col("payload_json.platform").alias(
        "platform"
    ),
    col("payload_json.parameter_name").alias(
        "parameter_name"
    ),
    col("payload_json.parameter_value").alias(
        "parameter_value"
    )

)

# --------------------------------------------------
# Create item_id
# --------------------------------------------------
# item_id is populated from parameter_value.
# Gold layer filters parameter_name='item_id'
# to identify valid item events.
# --------------------------------------------------

event_df = event_df.withColumn(

    "item_id",

    col("parameter_value")

)

# --------------------------------------------------
# Drop Unnecessary Columns
# --------------------------------------------------
# Temporary payload columns are removed
# after flattening.
# --------------------------------------------------

event_df = event_df.drop(

    "payload_json",
    "event_payload"

)

# --------------------------------------------------
# Data Validation Checks
# --------------------------------------------------
# Validation checks are performed before
# saving Silver tables.
# --------------------------------------------------

print("Silver Event Record Count:")

print(event_df.count())

print("Silver Event Schema:")

event_df.printSchema()

print("Flattened Payload Sample:")

event_df.select(

    "event_type",
    "platform",
    "parameter_name",
    "parameter_value",
    "item_id"

).show(10, truncate=False)

# --------------------------------------------------
# Save Silver Delta Tables
# --------------------------------------------------
# Cleaned and transformed datasets are
# stored as Delta tables.
# --------------------------------------------------

item_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_item")

event_df.write.format("delta") \
    .partitionBy("year") \
    .mode("overwrite").saveAsTable("silver_event")

# --------------------------------------------------
# Silver Layer Completion
# --------------------------------------------------

print("Silver layer completed successfully.")

print("Tables Created:")

print("- silver_item")

print("- silver_event")    